# Sri Lankan Monsoon Aviation Weather Analysis
This Jupyter Notebook contains the exploratory data analysis (EDA), Feature Engineering, and visualizations for the hyper-local aviation wind prediction model.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_regression
import warnings

warnings.filterwarnings('ignore')
plt.style.use('dark_background') # For premium look

# Load Data
df = pd.read_excel('../BIA_METAR_DATA_(2019_2024).xlsx')
df.head()

## 1. Data Cleaning & Preprocessing

In [ ]:
col_mapping = {
    'Wind Dir.': 'Wind Dir',
    'Dry tem(0C)': 'Dry Temp(0C)',
    'Dew point(0C)': 'Dew point(0C)', 
    'RH(%)': 'RH(%)',
    'QNH (hPa)': 'QNH(hPa)'
}
df.rename(columns=col_mapping, inplace=True)

# Filter for Monsoon Months (May to September)
df_monsoon = df[df['Month'].isin([5, 6, 7, 8, 9])].copy()

df_monsoon['Wind Dir'] = pd.to_numeric(df_monsoon['Wind Dir'], errors='coerce')
for col in ['Wind speed(Kts)', 'Dry Temp(0C)', 'Dew point(0C)', 'RH(%)', 'QNH(hPa)']:
    df_monsoon[col] = pd.to_numeric(df_monsoon[col], errors='coerce')

df_monsoon.ffill(inplace=True)
df_monsoon.dropna(subset=['Wind speed(Kts)'], inplace=True)

print(f"Total Monsoon Records: {len(df_monsoon)}")

## 2. Feature Engineering
Creating the advanced features required for high accuracy.

In [ ]:
# Circular Encoding
df_monsoon['Wind Dir_Sin'] = np.sin(np.radians(df_monsoon['Wind Dir']))
df_monsoon['Wind Dir_Cos'] = np.cos(np.radians(df_monsoon['Wind Dir']))

# Dew Point Depression
df_monsoon['Dew_Point_Depression'] = df_monsoon['Dry Temp(0C)'] - df_monsoon['Dew point(0C)']

# Time lags (Assuming 30min intervals, 6 rows = 3 hours)
df_monsoon['Wind speed(Kts)_lag_3h'] = df_monsoon['Wind speed(Kts)'].shift(6)
df_monsoon.dropna(inplace=True)

df_monsoon[['Wind speed(Kts)', 'Wind Dir_Sin', 'Wind speed(Kts)_lag_3h', 'Dew_Point_Depression']].head()

## 3. Mutual Information Scores
Discovering the best relations for the model.

In [ ]:
features = ['Wind Dir_Sin', 'Wind Dir_Cos', 'Wind speed(Kts)_lag_3h', 'Dew_Point_Depression', 'Dry Temp(0C)', 'RH(%)', 'QNH(hPa)']
target = 'Wind speed(Kts)'

X = df_monsoon[features]
y = df_monsoon[target]

mi_scores = mutual_info_regression(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=features).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=mi_series.values, y=mi_series.index, palette='viridis')
plt.title('Mutual Information Scores (Feature Importance)')
plt.show()

## 4. Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 8))
corr = df_monsoon[features + [target]].corr(method='spearman')
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Spearman Correlation Heatmap')
plt.show()